In [1]:
from bs4 import BeautifulSoup
import requests
import time
from nba_api.stats.endpoints import LeagueDashPlayerStats
import pandas as pd
import numpy as np

In [103]:
year=1997
url = f"https://www.basketball-reference.com/wnba/years/{year}_advanced.html"
response = requests.get(url)
soup = BeautifulSoup(response.content, 'html.parser')

table = soup.find('table', {'id':'advanced'})
headers =  ['Season']+[th.getText() for th in table.find('tr').find_all('th')]
rows = []
for row in table.find_all('tr', {'class':'full_table'}):
    ths = row.find_all('th')
    tds = row.find_all('td')
    rows.append([1997]+[th.getText() for th in ths]+[td.getText() for td in tds])

df = pd.DataFrame(rows, columns=headers)

In [104]:
df

,Season,Player,Team,Pos,G,MP,G,MP,PER,TS%,...,BLK%,TOV%,USG%,ORtg,DRtg,,OWS,DWS,WS,WS/40
0,1997,Anita Maxwell,CLE,F,9,63,9,63,10.6,.333,...,0.0,17.4,25.6,72,89,,-0.2,0.1,-0.1,-0.036
1,1997,Tammi Reiss,UTA,G,28,831,28,831,9.6,.423,...,0.2,19.3,17.0,87,105,,0.2,-0.7,-0.5,-0.024
2,1997,Catarina Pollini,HOU,F,13,94,13,94,4.6,.403,...,0.9,22.7,17.6,77,92,,-0.1,0.1,0.0,.005
3,1997,Bridget Pettis,PHO,G,28,842,28,842,17.0,.479,...,1.1,18.2,24.0,94,88,,1.5,1.7,3.2,.151
4,1997,Kim Perrot,HOU,G,28,692,28,692,14.1,.452,...,0.2,26.7,16.5,87,87,,0.2,1.5,1.7,.098
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,1997,Isabelle Fijalkowski,CLE,F-C,28,803,28,803,18.5,.570,...,1.8,20.0,21.2,105,94,,2.9,0.7,3.6,.180
93,1997,Michelle Edwards,CLE,G,20,622,20,622,14.4,.488,...,0.4,27.0,21.4,85,92,,0.0,0.7,0.8,.050
94,1997,Tamecka Dixon,LAS,G,27,715,27,715,19.2,.550,...,0.5,16.6,21.6,103,91,,2.3,1.0,3.3,.183
95,1997,Cassandra Crumpton-Moorer,NYL,G,2,11,2,11,3.0,.250,...,0.0,20.0,20.5,62,96,,0.0,0.0,0.0,-0.131


In [105]:
def get_advanced_stats_for_multiple_seasons(start=1997, end=2025):
    df = pd.DataFrame()
    for season in range(start, end+1):

        url = f"https://www.basketball-reference.com/wnba/years/{season}_advanced.html"
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')

        table = soup.find('table', {'id':'advanced'})
        raw_headers = [th.getText() for th in table.find('tr').find_all('th') if th.getText() not in ['Rk', 'Player']]
        headers =  ['Season', 'Player']+raw_headers

        rows = []
        for row in table.find_all('tr', {'class': 'full_table'}):
            player_cell = row.find(attrs={"data-stat": "player"})
            
            if player_cell:
                player_anchor = player_cell.find('a')
                player_name = player_anchor.getText().strip() if player_anchor else player_cell.getText().strip()
                
                stat_cells = [td.getText().strip() for td in row.find_all('td') if td.get('data-stat') != 'player']

                rows.append([season, player_name] + stat_cells)

        df = pd.concat([df, pd.DataFrame(rows, columns=headers)], axis=0)

        time.sleep(1)
    return df


In [138]:
scraped_advanced_stats = get_advanced_stats_for_multiple_seasons()
scraped_advanced_stats.shape

(4691, 27)

In [139]:
advanced_stats = scraped_advanced_stats
advanced_stats.tail()

,Season,Player,Team,Pos,G,MP,G,MP,PER,TS%,...,BLK%,TOV%,USG%,ORtg,DRtg,,OWS,DWS,WS,WS/40
177,2025,Brittney Sykes,TOT,G,39,1194,39,1194,13.4,.497,...,1.1,15.0,25.1,97,106,,0.0,1.4,1.4,.046
178,2025,Stephanie Talbot,TOT,F,38,541,38,541,9.2,.481,...,1.0,25.6,13.4,95,103,,-0.1,0.8,0.7,.054
179,2025,Sevgi Uzun,TOT,G,25,439,25,439,2.7,.382,...,0.6,29.6,14.8,74,113,,-1.2,0.0,-1.2,-0.105
180,2025,Julie Vanloo,TOT,G,35,449,35,449,3.5,.429,...,0.0,29.8,16.7,79,111,,-1.0,0.2,-0.9,-0.078
181,2025,Li Yueru,TOT,C,31,519,31,519,13.6,.534,...,1.7,14.6,17.3,108,111,,0.7,0.2,0.9,.071


In [140]:
advanced_stats.info()

<class 'pandas.DataFrame'>
Index: 4691 entries, 0 to 181
Data columns (total 27 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Season  4691 non-null   int64
 1   Player  4691 non-null   str  
 2   Team    4691 non-null   str  
 3   Pos     4691 non-null   str  
 4   G       4691 non-null   str  
 5   MP      4691 non-null   str  
 6   G       4691 non-null   str  
 7   MP      4691 non-null   str  
 8   PER     4691 non-null   str  
 9   TS%     4691 non-null   str  
 10  eFG%    4691 non-null   str  
 11  3PAr    4691 non-null   str  
 12  FTr     4691 non-null   str  
 13  ORB%    4691 non-null   str  
 14  TRB%    4691 non-null   str  
 15  AST%    4691 non-null   str  
 16  STL%    4691 non-null   str  
 17  BLK%    4691 non-null   str  
 18  TOV%    4691 non-null   str  
 19  USG%    4691 non-null   str  
 20  ORtg    4691 non-null   str  
 21  DRtg    4691 non-null   str  
 22          4691 non-null   str  
 23  OWS     4691 non-null   str  


In [141]:
advanced_stats.dtypes

Season    int64
Player      str
Team        str
Pos         str
G           str
MP          str
G           str
MP          str
PER         str
TS%         str
eFG%        str
3PAr        str
FTr         str
ORB%        str
TRB%        str
AST%        str
STL%        str
BLK%        str
TOV%        str
USG%        str
ORtg        str
DRtg        str
            str
OWS         str
DWS         str
WS          str
WS/40       str
dtype: object

In [142]:
advanced_stats.columns[-5]

'\xa0'

In [143]:
advanced_stats= advanced_stats.loc[:, ~advanced_stats.columns.duplicated()] # remove duplicated G and MP columns
advanced_stats.columns

Index(['Season', 'Player', 'Team', 'Pos', 'G', 'MP', 'PER', 'TS%', 'eFG%',
       '3PAr', 'FTr', 'ORB%', 'TRB%', 'AST%', 'STL%', 'BLK%', 'TOV%', 'USG%',
       'ORtg', 'DRtg', ' ', 'OWS', 'DWS', 'WS', 'WS/40'],
      dtype='str')

In [146]:
advanced_stats.drop(advanced_stats.columns[-5], axis=1, inplace=True) # remove ' ' column , use indices to avoid keyerror
advanced_stats.drop(columns=['Team', 'Pos', 'G', 'MP'], inplace=True) # remove 'Team', 'Pos', 'G', 'MP', 'G', 'MP' columns
advanced_stats.columns

Index(['Season', 'Player', 'PER', 'TS%', 'eFG%', '3PAr', 'FTr', 'ORB%', 'TRB%',
       'AST%', 'STL%', 'BLK%', 'TOV%', 'USG%', 'ORtg', 'DRtg', 'OWS', 'DWS',
       'WS', 'WS/40'],
      dtype='str')

In [147]:
decimal_cols = advanced_stats.columns[2:]

for col in decimal_cols:
    advanced_stats[col] = pd.to_numeric(advanced_stats[col], errors='coerce')


In [148]:
advanced_stats.info()

<class 'pandas.DataFrame'>
Index: 4691 entries, 0 to 181
Data columns (total 20 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Season  4691 non-null   int64  
 1   Player  4691 non-null   str    
 2   PER     4683 non-null   float64
 3   TS%     4653 non-null   float64
 4   eFG%    4646 non-null   float64
 5   3PAr    4646 non-null   float64
 6   FTr     4646 non-null   float64
 7   ORB%    4683 non-null   float64
 8   TRB%    4683 non-null   float64
 9   AST%    4683 non-null   float64
 10  STL%    4683 non-null   float64
 11  BLK%    4683 non-null   float64
 12  TOV%    4664 non-null   float64
 13  USG%    4683 non-null   float64
 14  ORtg    4665 non-null   float64
 15  DRtg    4683 non-null   float64
 16  OWS     4691 non-null   float64
 17  DWS     4691 non-null   float64
 18  WS      4691 non-null   float64
 19  WS/40   4683 non-null   float64
dtypes: float64(18), int64(1), str(1)
memory usage: 769.6 KB


In [ ]:
advanced_stats.to_csv('../data/wnba_advanced_players_stats.csv')

In [51]:
year=1997
url = f"https://www.basketball-reference.com/wnba/years/{year}_totals.html"
response = requests.get(url)
soup = BeautifulSoup(response.content, 'html.parser')

table = soup.find('table', {'id':'totals'})
headers =  ['Season']+[th.getText() for th in table.find('tr').find_all('th')]
rows = []
for row in table.find_all('tr', {'class':'full_table'}):
    ths = row.find_all('th')
    tds = row.find_all('td')
    rows.append([1997]+[th.getText() for th in ths]+[td.getText() for td in tds])

df = pd.DataFrame(rows, columns=headers)
df

,Season,Player,Team,Pos,G,MP,G,GS,MP,FG,...,FTA,FT%,ORB,TRB,AST,STL,BLK,TOV,PF,PTS
0,1997,Anita Maxwell,CLE,F,9,63,9,0,63,8,...,8,.375,2,12,8,4,0,6,5,19
1,1997,Tammi Reiss,UTA,G,28,831,28,26,831,72,...,55,.764,29,77,87,23,2,61,57,216
2,1997,Catarina Pollini,HOU,F,13,94,13,0,94,8,...,12,.500,3,12,5,4,1,8,21,22
3,1997,Bridget Pettis,PHO,G,28,842,28,28,842,107,...,108,.898,36,107,78,49,11,82,57,352
4,1997,Kim Perrot,HOU,G,28,692,28,24,692,59,...,37,.405,18,75,88,69,2,65,53,161
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,1997,Isabelle Fijalkowski,CLE,F-C,28,803,28,28,803,125,...,103,.786,62,157,68,17,17,73,129,332
93,1997,Michelle Edwards,CLE,G,20,622,20,14,622,76,...,86,.523,13,70,89,35,3,77,40,203
94,1997,Tamecka Dixon,LAS,G,27,715,27,21,715,115,...,88,.773,22,81,55,49,5,58,76,320
95,1997,Cassandra Crumpton-Moorer,NYL,G,2,11,2,0,11,1,...,0,,1,2,1,0,0,1,0,2


In [38]:
def get_basic_stats_for_multiple_seasons(start=1997, end=2025):
    df = pd.DataFrame()
    for season in range(start, end+1):

        url = f"https://www.basketball-reference.com/wnba/years/{season}_totals.html"
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')

        table = soup.find('table', {'id':'totals'})
        raw_headers = [th.getText() for th in table.find('tr').find_all('th') if th.getText() not in ['Rk', 'Player']]
        headers =  ['Season', 'Player']+raw_headers

        rows = []
        for row in table.find_all('tr', {'class': 'full_table'}):
            player_cell = row.find(attrs={"data-stat": "player"})
            
            if player_cell:
                player_anchor = player_cell.find('a')
                player_name = player_anchor.getText().strip() if player_anchor else player_cell.getText().strip()
                
                stat_cells = [td.getText().strip() for td in row.find_all('td') if td.get('data-stat') != 'player']

                rows.append([season, player_name] + stat_cells)

        df = pd.concat([df, pd.DataFrame(rows, columns=headers)], axis=0)

        time.sleep(1)
    return df

In [39]:
scraped_basic_stats = get_basic_stats_for_multiple_seasons()
scraped_basic_stats.shape

(4691, 29)

In [40]:
basic_stats = scraped_basic_stats
basic_stats.tail()

,Season,Player,Team,Pos,G,MP,G,GS,MP,FG,...,FTA,FT%,ORB,TRB,AST,STL,BLK,TOV,PF,PTS
177,2025,Brittney Sykes,TOT,G,39,1194,39,38,1194,174,...,219,.781,20,123,155,48,14,98,99,550
178,2025,Stephanie Talbot,TOT,F,38,541,38,10,541,39,...,28,.643,26,101,66,19,6,40,43,112
179,2025,Sevgi Uzun,TOT,G,25,439,25,3,439,31,...,16,.688,1,31,63,13,3,43,37,78
180,2025,Julie Vanloo,TOT,G,35,449,35,2,449,33,...,10,.800,4,32,68,15,0,49,45,99
181,2025,Li Yueru,TOT,C,31,519,31,12,519,62,...,50,.860,52,141,29,9,9,30,57,187


In [151]:
basic_stats= basic_stats.loc[:, ~basic_stats.columns.duplicated()] # remove duplicated column G MP
basic_stats.columns

Index(['Season', 'Player', 'Team', 'Pos', 'G', 'MP', 'GS', 'FG', 'FGA', 'FG%',
       '3P', '3PA', '3P%', '2P', '2PA', '2P%', 'FT', 'FTA', 'FT%', 'ORB',
       'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS'],
      dtype='str')

In [152]:
basic_stats.drop(columns=['Team', 'Pos'], inplace=True) # remove 'Team', 'Pos'columns
basic_stats.columns

Index(['Season', 'Player', 'G', 'MP', 'GS', 'FG', 'FGA', 'FG%', '3P', '3PA',
       '3P%', '2P', '2PA', '2P%', 'FT', 'FTA', 'FT%', 'ORB', 'TRB', 'AST',
       'STL', 'BLK', 'TOV', 'PF', 'PTS'],
      dtype='str')

In [153]:
basic_stats.info()

<class 'pandas.DataFrame'>
Index: 4691 entries, 0 to 181
Data columns (total 25 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Season  4691 non-null   int64
 1   Player  4691 non-null   str  
 2   G       4691 non-null   str  
 3   MP      4691 non-null   str  
 4   GS      4691 non-null   str  
 5   FG      4691 non-null   str  
 6   FGA     4691 non-null   str  
 7   FG%     4691 non-null   str  
 8   3P      4691 non-null   str  
 9   3PA     4691 non-null   str  
 10  3P%     4691 non-null   str  
 11  2P      4691 non-null   str  
 12  2PA     4691 non-null   str  
 13  2P%     4691 non-null   str  
 14  FT      4691 non-null   str  
 15  FTA     4691 non-null   str  
 16  FT%     4691 non-null   str  
 17  ORB     4691 non-null   str  
 18  TRB     4691 non-null   str  
 19  AST     4691 non-null   str  
 20  STL     4691 non-null   str  
 21  BLK     4691 non-null   str  
 22  TOV     4691 non-null   str  
 23  PF      4691 non-null   str  


In [154]:
cols_to_convert = basic_stats.columns[2:]

for col in cols_to_convert:
    basic_stats[col] = pd.to_numeric(basic_stats[col], errors='coerce')

basic_stats.info()

<class 'pandas.DataFrame'>
Index: 4691 entries, 0 to 181
Data columns (total 25 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Season  4691 non-null   int64  
 1   Player  4691 non-null   str    
 2   G       4691 non-null   int64  
 3   MP      4691 non-null   int64  
 4   GS      4691 non-null   int64  
 5   FG      4691 non-null   int64  
 6   FGA     4691 non-null   int64  
 7   FG%     4646 non-null   float64
 8   3P      4691 non-null   int64  
 9   3PA     4691 non-null   int64  
 10  3P%     3919 non-null   float64
 11  2P      4691 non-null   int64  
 12  2PA     4691 non-null   int64  
 13  2P%     4623 non-null   float64
 14  FT      4691 non-null   int64  
 15  FTA     4691 non-null   int64  
 16  FT%     4422 non-null   float64
 17  ORB     4691 non-null   int64  
 18  TRB     4691 non-null   int64  
 19  AST     4691 non-null   int64  
 20  STL     4691 non-null   int64  
 21  BLK     4691 non-null   int64  
 22  TOV     4691 non-

In [ ]:
basic_stats.to_csv('../data/wnba_players_basic_stats.csv')

In [157]:
url = "https://www.basketball-reference.com/wnba/awards/mvp.html"
response = requests.get(url)
soup = BeautifulSoup(response.content, 'html.parser')
table = soup.find('table', {'id':'mvp'})
headers =  [th.getText() for th in table.find('tr', {'over_header'}).find_next_sibling().find_all('th')]
headers
rows = []
for row in table.find_all('tr'):
    ths = row.find_all('th')
    tds = row.find_all('td')
    rows.append([th.getText() for th in ths]+[td.getText() for td in tds])

df = pd.DataFrame(rows, columns=headers)
df

,Year,Player,Voting,Tm,G,MP,PTS,TRB,AST,STL,BLK,FG%,3P%,FT%,WS,WS/48
0,,,Per Game,Shooting,Advanced,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Year,Player,Voting,Tm,G,MP,PTS,TRB,AST,STL,BLK,FG%,3P%,FT%,WS,WS/48
2,2025,A'ja Wilson,(V),LVA,40,31.2,23.4,10.2,3.1,1.6,2.3,.505,.424,.855,9.5,.365
3,2024,A'ja Wilson,(V),LVA,38,34.4,26.9,11.9,2.3,1.8,2.6,.518,.317,.844,10.9,.399
4,2023,Breanna Stewart,(V),NYL,40,34.1,23.0,9.3,3.8,1.5,1.6,.465,.355,.851,10.3,.362
5,2022,A'ja Wilson,(V),LVA,36,30.0,19.5,9.4,2.1,1.4,1.9,.501,.373,.813,6.5,.289
6,2021,Jonquel Jones,(V),CON,27,31.7,19.4,11.2,2.8,1.3,1.3,.515,.362,.802,6.6,.369
7,2020,A'ja Wilson,(V),LVA,22,31.7,20.5,8.5,2.0,1.2,2.0,.480,,.781,4.0,.276
8,2019,Elena Delle Donne,(V),WAS,31,29.1,19.5,8.2,2.2,0.6,1.3,.515,.430,.974,7.7,.411
9,2018,Breanna Stewart,(V),SEA,34,31.6,21.8,8.4,2.5,1.4,1.4,.529,.415,.820,7.7,.346


In [158]:
mvps = df.iloc[2:,:].reset_index().drop(columns='index')
mvps

,Year,Player,Voting,Tm,G,MP,PTS,TRB,AST,STL,BLK,FG%,3P%,FT%,WS,WS/48
0,2025,A'ja Wilson,(V),LVA,40,31.2,23.4,10.2,3.1,1.6,2.3,.505,.424,.855,9.5,.365
1,2024,A'ja Wilson,(V),LVA,38,34.4,26.9,11.9,2.3,1.8,2.6,.518,.317,.844,10.9,.399
2,2023,Breanna Stewart,(V),NYL,40,34.1,23.0,9.3,3.8,1.5,1.6,.465,.355,.851,10.3,.362
3,2022,A'ja Wilson,(V),LVA,36,30.0,19.5,9.4,2.1,1.4,1.9,.501,.373,.813,6.5,.289
4,2021,Jonquel Jones,(V),CON,27,31.7,19.4,11.2,2.8,1.3,1.3,.515,.362,.802,6.6,.369
5,2020,A'ja Wilson,(V),LVA,22,31.7,20.5,8.5,2.0,1.2,2.0,.480,,.781,4.0,.276
6,2019,Elena Delle Donne,(V),WAS,31,29.1,19.5,8.2,2.2,0.6,1.3,.515,.430,.974,7.7,.411
7,2018,Breanna Stewart,(V),SEA,34,31.6,21.8,8.4,2.5,1.4,1.4,.529,.415,.820,7.7,.346
8,2017,Sylvia Fowles,(V),MIN,34,30.8,18.9,10.4,1.5,1.3,2.0,.655,,.768,9.2,.420
9,2016,Nneka Ogwumike,(V),LAS,33,31.6,19.7,9.1,3.1,1.3,1.2,.665,.615,.869,9.6,.444


In [159]:
mvps.info()

<class 'pandas.DataFrame'>
RangeIndex: 29 entries, 0 to 28
Data columns (total 16 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Year    29 non-null     str  
 1   Player  29 non-null     str  
 2   Voting  29 non-null     str  
 3   Tm      29 non-null     str  
 4   G       29 non-null     str  
 5   MP      29 non-null     str  
 6   PTS     29 non-null     str  
 7   TRB     29 non-null     str  
 8   AST     29 non-null     str  
 9   STL     29 non-null     str  
 10  BLK     29 non-null     str  
 11  FG%     29 non-null     str  
 12  3P%     29 non-null     str  
 13  FT%     29 non-null     str  
 14  WS      29 non-null     str  
 15  WS/48   29 non-null     str  
dtypes: str(16)
memory usage: 3.8 KB


In [160]:
mvps.columns

Index(['Year', 'Player', 'Voting', 'Tm', 'G', 'MP', 'PTS', 'TRB', 'AST', 'STL',
       'BLK', 'FG%', '3P%', 'FT%', 'WS', 'WS/48'],
      dtype='str')

In [161]:
cols_to_convert = ['Year', 'G', 'MP', 'PTS', 'TRB', 'AST', 'STL','BLK', 'FG%', '3P%', 'FT%', 'WS', 'WS/48']

for col in cols_to_convert:
    mvps[col] = pd.to_numeric(mvps[col], errors='coerce')

In [162]:
mvps.info()

<class 'pandas.DataFrame'>
RangeIndex: 29 entries, 0 to 28
Data columns (total 16 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Year    29 non-null     int64  
 1   Player  29 non-null     str    
 2   Voting  29 non-null     str    
 3   Tm      29 non-null     str    
 4   G       29 non-null     int64  
 5   MP      29 non-null     float64
 6   PTS     29 non-null     float64
 7   TRB     29 non-null     float64
 8   AST     29 non-null     float64
 9   STL     29 non-null     float64
 10  BLK     29 non-null     float64
 11  FG%     29 non-null     float64
 12  3P%     27 non-null     float64
 13  FT%     29 non-null     float64
 14  WS      29 non-null     float64
 15  WS/48   29 non-null     float64
dtypes: float64(11), int64(2), str(3)
memory usage: 3.8 KB


In [45]:
mvps.to_csv('../data/wnba_mvps.csv')

In [22]:
url = "https://www.basketball-reference.com/wnba/awards/awards_1997.html#all_mvp"
response = requests.get(url)
soup = BeautifulSoup(response.content, 'html.parser')
table = soup.find('table', {'id':'mvp'})
headers =  [th.getText() for th in table.find('tr', {'over_header'}).find_next_sibling().find_all('th')]
rows = []
for row in table.find('tbody').find_all('tr'):
    ths = row.find_all('th')
    tds = row.find_all('td')
    rows.append([th.getText() for th in ths]+[td.getText() for td in tds])

df = pd.DataFrame(rows, columns=headers)
column_names = list(df.columns)
print(column_names)
df.head()

['Rank', 'Player', 'Age', 'Tm', 'First', 'Pts Won', 'Pts Max', 'Share', 'G', 'MP', 'PTS', 'TRB', 'AST', 'STL', 'BLK', 'FG%', '3P%', 'FT%', 'WS', 'WS/48']


,Rank,Player,Age,Tm,First,Pts Won,Pts Max,Share,G,MP,PTS,TRB,AST,STL,BLK,FG%,3P%,FT%,WS,WS/48
0,1,Cynthia Cooper,33,HOU,37,370,370,1,28,35.1,22.2,4.0,4.7,2.1,0.2,.470,.414,.864,9.4,.462
1,2,Andrea Stinson,29,CHA,0,116,370,0.314,28,36.1,15.7,5.5,4.4,1.5,0.8,.447,.325,.674,3.8,.182
2,3,Lisa Leslie,24,LAS,0,109,370,0.295,28,32.2,15.9,9.5,2.6,1.4,2.1,.431,.261,.598,3.5,.189
3,4,Ruthie Bolton,29,SAC,0,107,370,0.289,23,35.3,19.4,5.8,2.6,2.3,0.0,.402,.344,.768,3.8,.224
4,5,Michele Timms,31,PHO,0,20,370,0.054,27,35.8,12.1,3.7,5.0,2.6,0.1,.336,.345,.760,3.9,.195


In [ ]:
def extract_all_mvp_votes(start_year=1997, end_year=2025):
    df = pd.DataFrame()
    for year in range(start_year,end_year+1):
        url = f"https://www.basketball-reference.com/wnba/awards/awards_{year}.html#all_mvp"
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        table = soup.find('table', {'id':'mvp'})
        headers =  ['Season']+[th.getText() for th in table.find('tr', {'over_header'}).find_next_sibling().find_all('th')]
        rows = []
        for row in table.find('tbody').find_all('tr'):
            ths = row.find_all('th')
            tds = row.find_all('td')
            rows.append([year]+[th.getText() for th in ths]+[td.getText() for td in tds])

        data = pd.DataFrame(rows, columns=headers)
        df = pd.concat([df, data], axis=0)

        time.sleep(4)
    return df


In [ ]:
scraped_mvp_votes = extract_all_mvp_votes()

In [29]:
scraped_mvp_votes.shape

(411, 21)

In [30]:
mvp_votes = scraped_mvp_votes
mvp_votes.head()

,Season,Rank,Player,Age,Tm,First,Pts Won,Pts Max,Share,G,...,PTS,TRB,AST,STL,BLK,FG%,3P%,FT%,WS,WS/48
0,1997,1,Cynthia Cooper,33,HOU,37,370,370,1,28,...,22.2,4.0,4.7,2.1,0.2,.470,.414,.864,9.4,.462
1,1997,2,Andrea Stinson,29,CHA,0,116,370,0.314,28,...,15.7,5.5,4.4,1.5,0.8,.447,.325,.674,3.8,.182
2,1997,3,Lisa Leslie,24,LAS,0,109,370,0.295,28,...,15.9,9.5,2.6,1.4,2.1,.431,.261,.598,3.5,.189
3,1997,4,Ruthie Bolton,29,SAC,0,107,370,0.289,23,...,19.4,5.8,2.6,2.3,0.0,.402,.344,.768,3.8,.224
4,1997,5,Michele Timms,31,PHO,0,20,370,0.054,27,...,12.1,3.7,5.0,2.6,0.1,.336,.345,.760,3.9,.195


In [32]:
print(mvp_votes.columns)
mvp_votes.info()

Index(['Season', 'Rank', 'Player', 'Age', 'Tm', 'First', 'Pts Won', 'Pts Max',
       'Share', 'G', 'MP', 'PTS', 'TRB', 'AST', 'STL', 'BLK', 'FG%', '3P%',
       'FT%', 'WS', 'WS/48'],
      dtype='str')
<class 'pandas.DataFrame'>
Index: 411 entries, 0 to 12
Data columns (total 21 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   Season   411 non-null    int64
 1   Rank     411 non-null    str  
 2   Player   411 non-null    str  
 3   Age      411 non-null    str  
 4   Tm       411 non-null    str  
 5   First    411 non-null    str  
 6   Pts Won  411 non-null    str  
 7   Pts Max  411 non-null    str  
 8   Share    411 non-null    str  
 9   G        411 non-null    str  
 10  MP       411 non-null    str  
 11  PTS      411 non-null    str  
 12  TRB      411 non-null    str  
 13  AST      411 non-null    str  
 14  STL      411 non-null    str  
 15  BLK      411 non-null    str  
 16  FG%      411 non-null    str  
 17  3P%      411 non-nu

In [33]:
cols_to_convert = ['Age', 'First', 'Pts Won', 'Pts Max',
       'Share', 'G', 'MP', 'PTS', 'TRB', 'AST', 'STL', 'BLK', 'FG%', '3P%',
       'FT%', 'WS', 'WS/48']
for col in cols_to_convert:
    mvp_votes[col] = pd.to_numeric(mvp_votes[col], errors='coerce')
mvp_votes.info()

<class 'pandas.DataFrame'>
Index: 411 entries, 0 to 12
Data columns (total 21 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Season   411 non-null    int64  
 1   Rank     411 non-null    str    
 2   Player   411 non-null    str    
 3   Age      411 non-null    int64  
 4   Tm       411 non-null    str    
 5   First    249 non-null    float64
 6   Pts Won  411 non-null    float64
 7   Pts Max  411 non-null    int64  
 8   Share    411 non-null    float64
 9   G        411 non-null    int64  
 10  MP       411 non-null    float64
 11  PTS      411 non-null    float64
 12  TRB      411 non-null    float64
 13  AST      411 non-null    float64
 14  STL      411 non-null    float64
 15  BLK      411 non-null    float64
 16  FG%      411 non-null    float64
 17  3P%      388 non-null    float64
 18  FT%      411 non-null    float64
 19  WS       411 non-null    float64
 20  WS/48    411 non-null    float64
dtypes: float64(14), int64(4), str(3)


In [41]:
mvp_votes.rename(columns={
    'Share':'award_share'
}, inplace=True)

In [42]:
mvp_votes.to_csv(r'../data/wnba_mvp_votes.csv')

In [8]:
# extract plus minus feature from nba api
stats = ['Season','PLAYER_NAME','PLUS_MINUS']
def extract_plus_minus_for_multiple_seasons(start_season=1997, end_season=2025, stats=stats):
    
    df = pd.DataFrame(columns=stats)
    for season in range(start_season, end_season+1):
        data = LeagueDashPlayerStats(league_id_nullable=10, season=str(season)).get_data_frames()[0]
        data['Season'] = season

        df = pd.concat([df, data.loc[:, stats]], axis=0)

        time.sleep(4)
        
    return df

In [9]:
scraped_bpm = extract_plus_minus_for_multiple_seasons()


In [11]:
data_bpm = scraped_bpm
data_bpm.tail()

,Season,PLAYER_NAME,PLUS_MINUS
177,2025,Tina Charles,-406
178,2025,Tyasha Harris,15
179,2025,Veronica Burton,97
180,2025,Yvonne Anderson,13
181,2025,Zia Cooke,-69


In [13]:
data_bpm.shape

(4692, 3)

In [15]:
data_bpm[['Season','PLAYER_NAME']].value_counts()

Season  PLAYER_NAME      
2002    Claudia Neves        2
1997    Adrienne Johnson     1
        Andrea Congreaves    1
        Andrea Stinson       1
        Anita Maxwell        1
                            ..
2025    Tina Charles         1
        Tyasha Harris        1
        Veronica Burton      1
        Yvonne Anderson      1
        Zia Cooke            1
Name: count, Length: 4691, dtype: int64

In [16]:
data_bpm[(data_bpm['Season']==2002)&(data_bpm['PLAYER_NAME']=='Claudia Neves')]

,Season,PLAYER_NAME,PLUS_MINUS
39,2002,Claudia Neves,-8
40,2002,Claudia Neves,0


In [17]:
data_bpm.rename(columns={
    'PLAYER_NAME':'Player'
}, inplace=True)
data_bpm.columns

Index(['Season', 'Player', 'PLUS_MINUS'], dtype='str')

In [44]:
data_bpm = data_bpm.groupby(['Season', 'Player'], as_index=False)['PLUS_MINUS'].sum()
data_bpm[(data_bpm['Season'] == 2002) & (data_bpm['Player'] == 'Claudia Neves')]

,Season,Player,PLUS_MINUS
824,2002,Claudia Neves,-8


In [45]:
data_bpm.to_csv(r'../data/wnba_players_plus_minus.csv')